In [1]:
import string
import pandas as pd

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tag import pos_tag
from nltk.probability import FreqDist
from nltk.classify import NaiveBayesClassifier, accuracy

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import spacy
import pickle
import os


/opt/homebrew/Caskroom/miniforge/base/envs/ai_core/lib/python3.10/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


In [2]:
MODEL_PATH = 'models_pt11/model.pickle'
DATA_PATH = 'dataset/jobpostingdata.csv'

# Sentimen Analysis

In [3]:
lemmatizer = WordNetLemmatizer()

In [4]:
def get_label(tag):
    if tag == 'jj':
        return 'a'
    elif tag in ['rb','vb','nn']:
        return tag[0]
    else:
        return 'n'


def lemmatizing(word_list):
    lemma_list = []
    tagged = pos_tag(word_list)
    for word, tag in tagged:
        label = get_label(tag.lower())
        if label:
            result = lemmatizer.lemmatize(word, label)
            lemma_list.append(result)
        else:
            result = lemmatizer.lemmatize(word)
            lemma_list.append(result)
    return lemma_list


def preprocessing(sentence):
    #tokenizer
    word_list = word_tokenize(sentence)

    # lowering
    word_list = [word.lower() for word in word_list]

    # Remove Stopwords and Number
    eng_stopwords = stopwords.words('english')
    removed_punc = [word for word in word_list if word not in eng_stopwords and word.isalpha()]

    # Stemming
    # stemmer = SnowballStemmer('english')
    # stemmed = [stemmer.stem(word) for word in removed_punc]

    # Lemmatizing + Post Tag
    lemmatized = lemmatizing(removed_punc)

    return lemmatized

In [5]:
# Read data
df = pd.read_csv(DATA_PATH)

X = df['text']
df['fraudulent'] = df['fraudulent'].map({0: 'Real Job Post', 1: 'Fake Job Post'})
Y = df['fraudulent']

all_sentence = ' '.join(X)
all_token = preprocessing(all_sentence)

# Cek frekuensi kata
freq_dist = FreqDist(all_token)
print(freq_dist.most_common(10))

[('work', 1046), ('experience', 964), ('team', 757), ('service', 756), ('customer', 732), ('company', 710), ('product', 640), ('business', 546), ('skill', 519), ('project', 518)]


In [6]:
def extract_features(text):
    features = {}
    for word in freq_dist.keys():
        features[word] = (word in text)
    return features

features_sets = []
for text, fraudulent in zip(X, Y):
    features = extract_features(preprocessing(text))
    features_sets.append((features, fraudulent))

print(features_sets[0])

({'php': True, 'developer': True, 'skilled': True, 'build': True, 'release': True, 'improve': True, 'grow': True, 'every': True, 'day': True, 'numerous': True, 'challenge': True, 'take': True, 'next': True, 'step': True, 'know': True, 'important': True, 'time': True, 'get': True, 'person': True, 'match': True, 'top': True, 'notch': True, 'company': True, 'fit': True, 'like': True, 'glove': True, 'kind': True, 'potential': True, 'make': True, 'sure': True, 'made': True, 'heaven': True, 'talk': True, 'first': True, 'test': True, 'skill': True, 'level': True, 'see': True, 'want': True, 'career': True, 'introduce': True, 'question': True, 'sound': True, 'familiar': True, 'answer': True, 'introducing': True, 'doingcoming': True, 'new': True, 'functionality': True, 'building': True, 'technology': True, 'run': True, 'web': True, 'scrum': True, 'team': True, 'specialization': True, 'also': True, 'able': True, 'perform': True, 'task': True, 'within': True, 'prioritize': True, 'work': True, 'tog

In [7]:
import random
random.shuffle(features_sets)
train_count = int(len(features_sets) * 0.8) # ambil 80% data untuk training
train_set = features_sets[:train_count]
test_set = features_sets[train_count:]

print("All Data: ", len(features_sets))
print("Train Data: ", len(train_set))
print("Test Data: ", len(test_set))

All Data:  500
Train Data:  400
Test Data:  100


In [8]:
def train_model():
    classifier = NaiveBayesClassifier.train(train_set)
    test_acc = accuracy(classifier, test_set)

    classifier.most_informative_features(10)
    print("Accuracy: ", test_acc*100)

    with open(MODEL_PATH, 'wb') as f:
        pickle.dump(classifier, f)
    return classifier

In [9]:
def analyze_statement(user_text, classifier):
    preprocessed = preprocessing(user_text)
    extracted = extract_features(preprocessed)
    prediction = classifier.classify(extracted)
    return prediction

In [10]:
def write_statement():
    while True:
        statement = input("Write your text")
        if len(statement) < 20:
            print("invalid must be at least 20 characters")
        elif len(statement.split()) <= 3:
            print("invalid must be at least 3 words")
        else:
            break
    return statement

# Menu

In [11]:
curr_text = None
curr_category = None
classifier = None

while True:
    print(f"\n{'='*60} \nRil or Fek Menu \n{'='*60}")
    print("Job Posting Classification")
    print("Real or Fake job posting classification & job recommendation based on text")
    print(f"your text: {curr_text}")
    print(f"your category: {curr_category}")
    print("1. Write Your Text")
    print("2. View Job Posting Recommendation Based on Text")
    print("3. View NER")
    print("4. Exit")

    choice = input("Enter your choice: ")
    if choice == '1':
        curr_text = write_statement()
        print("Your Text Right Now : ",curr_text)
        if os.path.exists(MODEL_PATH):
            print("Model found. Loading the model...")
            with open(MODEL_PATH, 'rb') as f:
                classifier = pickle.load(f)
        else:
            print("Model not found. Training a new model...")
            classifier = train_model()

        curr_category = analyze_statement(curr_text, classifier)
        print(f"Your text is classified as: {curr_category}")
        continue
        input("\nPress Enter to return to the menu...")

    elif choice == '2':
        if curr_text is None:
            print("Please enter your text first.")
            continue
        # TF-IDF Vectorization
        vectorizer = TfidfVectorizer(stop_words='english')

        corpus = df['text']
        tfidf_matrix = vectorizer.fit_transform(corpus)

        user_vec = vectorizer.transform({' '.join(word_tokenize(curr_text.lower()))})
        similarity_scores = cosine_similarity(user_vec, tfidf_matrix).flatten()
        # flatten -> utk ngubah bentuk similarity dari 2d jadi 1d [[0.1, 0.2, 0.3]] -> [0.1, 0.2, 0.3]
        top_job = similarity_scores.argsort()[-5:][::-1] # ambil 5 index dengan skor tertinggi
        for i, idx in enumerate(top_job):
            print(f"{i}. {df.iloc[idx]['title']}")
        
        input("\nPress Enter to return to the menu...")

    elif choice == '3':
        if curr_text is None:
            print("Please enter your text first.")
            continue
        nlp = spacy.load("en_core_web_sm")
        doc = nlp(curr_text)

        categories = {}

        print("Named Entities in your text:")
        for ent in doc.ents:
            if label not in categories:
                categories[label] = []
            categories[label].append(ent.text)
        
        for label, ents in categories.items():
            print(f"{label}: {', '.join(ents)}")

        input("\nPress Enter to return to the menu...")
            
    elif choice == '4':
        print("Exiting the program. Goodbye!")
        break
    else:
        print("Invalid choice. Please try again.")


Ril or Fek Menu 
Job Posting Classification
Real or Fake job posting classification & job recommendation based on text
your text: None
your category: None
1. Write Your Text
2. View Job Posting Recommendation Based on Text
3. View NER
4. Exit
Your Text Right Now :  Visit our website at www.savantdegrees.com to explore our exciting innovations. We're seeking a talented software developer to join our Asia team. This full-time position offers a remote working arrangement based in Indonesia.  Skills & Experience Needed  Minimum 3 years of current, hands-on experience in software development   Bachelors or Masters degree in Computer Science, Information Systems, or equivalent  Strong Java, HTML, CSS, JavaScript skills, with knowledge on Groovy, Apache Tomcat, Python, and JSON an added advantage  Experience with AI-powered development tools is a plus, including integrating AI coding assistants (e.g., Codex), and performing AI-assisted code review to improve development workflows and code qu

NameError: name 'label' is not defined